# v25 — улучшаем mAP, кандидатов v24 не меняем

**Restart Kernel → Run All** в прежнем research `.venv`. 84 фиксированных варианта, без обучения, GPU и интернета. Общего ограничения времени нет.

Выбор — только на calibration. На validation — исходный v24 и один победитель. Это уже наблюдавшиеся development-данные, не независимый тест. Активный MVP автоматически не меняется.


In [1]:
from pathlib import Path
import os, sys, subprocess
candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/map_search.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из исследовательского Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import Markdown, display
from training import map_search as experiment
SOURCE_RUN = 'dual_role_v1'
RUN_NAME = 'map_search_v1'
ALLOW_OUTER_EVALUATION = True
print('Kernel:', sys.executable, '\nRun:', RUN_NAME, '| experiments:', len(experiment.systems()))
print('CPU, cached features only; обучение: нет; кандидаты и порог R1: фиксированы')


Kernel: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python 
Run: map_search_v1 | experiments: 84
CPU, cached features only; обучение: нет; кандидаты и порог R1: фиксированы


## 1. Проверки
Независимость запросов, воспроизведение исходного ranking, неизменность R1-кандидата и выбор до validation.


In [2]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_map_search.py', 'tests/test_dual_role_experiment.py', 'tests/test_dual_role_inference.py'], check=True)


..............................                                           [100%]
30 passed in 4.30s


CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_map_search.py', 'tests/test_dual_role_experiment.py', 'tests/test_dual_role_inference.py'], returncode=0)

## 2. Preflight
Проверяем завершённый v24, фиксируем сетку и исходники. Ничего не обучаем и не исправляем в данных.


In [3]:
context = experiment.prepare(RUN_NAME, source_run=SOURCE_RUN)
print('Output:', context['output'])
display(context['manifest']['map_search_plan'])


Output: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_25_map_search/runs/map_search_v1


{'source_signature': '0723affdca3877377ab6d134a7c6e8e665181173bc9bddce8f2ff4d6a5d0da5f',
 'source_directory': '/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_24_dual_role/runs/dual_role_v1',
 'systems': [{'name': 'r1w00_k20_q3_l50',
   'k1': 20,
   'k2': 3,
   'lambda': 0.5,
   'r1_weight': 0.0},
  {'name': 'r1w00_k10_q1_l25',
   'k1': 10,
   'k2': 1,
   'lambda': 0.25,
   'r1_weight': 0.0},
  {'name': 'r1w00_k10_q1_l40',
   'k1': 10,
   'k2': 1,
   'lambda': 0.4,
   'r1_weight': 0.0},
  {'name': 'r1w00_k10_q1_l50',
   'k1': 10,
   'k2': 1,
   'lambda': 0.5,
   'r1_weight': 0.0},
  {'name': 'r1w00_k10_q1_l65',
   'k1': 10,
   'k2': 1,
   'lambda': 0.65,
   'r1_weight': 0.0},
  {'name': 'r1w00_k10_q1_l75',
   'k1': 10,
   'k2': 1,
   'lambda': 0.75,
   'r1_weight': 0.0},
  {'name': 'r1w00_k10_q1_l90',
   'k1': 10,
   'k2': 1,
   'lambda': 0.9,
   'r1_weight': 0.0},
  {'name': 'r1w00_k10_q3_l25',
   'k1': 10,
   'k2': 3,
 

## 3. Все эксперименты
Отображаются номер варианта из 84, параметры, mAP и время. Завершённые этапы сохраняются и при повторе отмечаются CACHED.


In [4]:
results = experiment.run(context, allow_outer=ALLOW_OUTER_EVALUATION)
print('Status:', results['status'], '| promoted:', results['promoted'])



CALIBRATION 1/84
[START] calibration_r1w00_k20_q3_l50 | elapsed 0.00 h
calibration r1w00_k20_q3_l50: mAP=0.787185; R1 candidates unchanged
[DONE] calibration_r1w00_k20_q3_l50 | 0.0 min

CALIBRATION 2/84
[START] calibration_r1w00_k10_q1_l25 | elapsed 0.00 h
calibration r1w00_k10_q1_l25: mAP=0.716427; R1 candidates unchanged
[DONE] calibration_r1w00_k10_q1_l25 | 0.0 min

CALIBRATION 3/84
[START] calibration_r1w00_k10_q1_l40 | elapsed 0.00 h
calibration r1w00_k10_q1_l40: mAP=0.731109; R1 candidates unchanged
[DONE] calibration_r1w00_k10_q1_l40 | 0.0 min

CALIBRATION 4/84
[START] calibration_r1w00_k10_q1_l50 | elapsed 0.00 h
calibration r1w00_k10_q1_l50: mAP=0.740008; R1 candidates unchanged
[DONE] calibration_r1w00_k10_q1_l50 | 0.0 min

CALIBRATION 5/84
[START] calibration_r1w00_k10_q1_l65 | elapsed 0.00 h
calibration r1w00_k10_q1_l65: mAP=0.749934; R1 candidates unchanged
[DONE] calibration_r1w00_k10_q1_l65 | 0.0 min

CALIBRATION 6/84
[START] calibration_r1w00_k10_q1_l75 | elapsed 0.00 

## 4. Итог
Сравниваем mAP выбранного варианта и v24. F1/TNR/C обязаны остаться прежними. Не выбираем новый вариант после просмотра validation.


In [5]:
display(Markdown((context['output'] / 'REPORT.md').read_text()))
print('Reports and exports:', context['output'])


# v25 — поиск mAP при фиксированных кандидатах v24

Статус: complete
84 заранее фиксированных варианта: 72 настройки MVP + 12 смесей MVP/full-train R1.
Выбор только по calibration mAP; при равенстве сохраняется исходный порядок (MVP первый).
Никакого обучения, нового извлечения признаков, подбора порога или изменения кандидатов.

| Validation: контроль и выбор | mAP@10 | F1 | TNR | C |
|---|---:|---:|---:|---:|
| r1w00_k20_q3_l50 | 0.814689 | 0.795455 | 0.709677 | 0.769721 |
| r1w50_k20_q3_l50 | 0.828957 | 0.795455 | 0.709677 | 0.769721 |

ΔmAP выбора против v24: +0.014269. Улучшено / ухудшено / равно AP: 34 / 21 / 192.

## Calibration: вся фиксированная сетка

| Вариант | mAP@10 |
|---|---:|
| r1w50_k20_q3_l50 | 0.826410 |
| r1w50_k20_q3_l65 | 0.825913 |
| r1w50_k20_q3_l40 | 0.821135 |
| r1w50_k20_q3_l75 | 0.821088 |
| r1w25_k20_q3_l65 | 0.814330 |
| r1w25_k20_q3_l50 | 0.813267 |
| r1w25_k20_q3_l40 | 0.811528 |
| r1w25_k20_q3_l75 | 0.808657 |
| r1w10_k20_q3_l40 | 0.797027 |
| r1w10_k20_q3_l50 | 0.795274 |
| r1w10_k20_q3_l65 | 0.794626 |
| r1w10_k20_q3_l75 | 0.790559 |
| r1w00_k20_q3_l50 | 0.787185 |
| r1w00_k10_q3_l65 | 0.786391 |
| r1w00_k20_q3_l40 | 0.785067 |
| r1w00_k20_q3_l65 | 0.785039 |
| r1w00_k40_q3_l65 | 0.784458 |
| r1w00_k40_q3_l50 | 0.784355 |
| r1w00_k10_q3_l50 | 0.784350 |
| r1w00_k10_q3_l75 | 0.784043 |
| r1w00_k20_q3_l25 | 0.783918 |
| r1w00_k40_q3_l40 | 0.783898 |
| r1w00_k10_q3_l25 | 0.782714 |
| r1w00_k10_q3_l40 | 0.782687 |
| r1w00_k20_q3_l75 | 0.779613 |
| r1w00_k40_q3_l25 | 0.778756 |
| r1w00_k40_q3_l75 | 0.778570 |
| r1w00_k30_q3_l50 | 0.776009 |
| r1w00_k30_q3_l40 | 0.774754 |
| r1w00_k10_q6_l75 | 0.774519 |
| r1w00_k30_q3_l65 | 0.773824 |
| r1w00_k10_q3_l90 | 0.773668 |
| r1w00_k30_q3_l75 | 0.772055 |
| r1w00_k20_q6_l75 | 0.770946 |
| r1w00_k20_q6_l65 | 0.769361 |
| r1w00_k20_q3_l90 | 0.768958 |
| r1w00_k10_q6_l90 | 0.768890 |
| r1w00_k20_q6_l90 | 0.768818 |
| r1w00_k10_q6_l65 | 0.768266 |
| r1w00_k30_q6_l75 | 0.767535 |
| r1w00_k30_q6_l65 | 0.767347 |
| r1w00_k40_q6_l75 | 0.767268 |
| r1w00_k30_q3_l25 | 0.767264 |
| r1w00_k40_q6_l65 | 0.765989 |
| r1w00_k10_q1_l90 | 0.765246 |
| r1w00_k10_q6_l50 | 0.765131 |
| r1w00_k40_q3_l90 | 0.764777 |
| r1w00_k30_q6_l90 | 0.764102 |
| r1w00_k30_q3_l90 | 0.763267 |
| r1w00_k40_q6_l90 | 0.762843 |
| r1w00_k20_q1_l90 | 0.760510 |
| r1w00_k40_q1_l90 | 0.759133 |
| r1w00_k20_q6_l50 | 0.758655 |
| r1w00_k10_q1_l75 | 0.756406 |
| r1w00_k30_q6_l50 | 0.755740 |
| r1w00_k10_q6_l40 | 0.755553 |
| r1w00_k30_q1_l90 | 0.754728 |
| r1w00_k20_q6_l40 | 0.751476 |
| r1w00_k40_q6_l50 | 0.750830 |
| r1w00_k10_q1_l65 | 0.749934 |
| r1w00_k10_q6_l25 | 0.746785 |
| r1w00_k20_q1_l75 | 0.746400 |
| r1w00_k30_q6_l40 | 0.746271 |
| r1w00_k40_q6_l40 | 0.746254 |
| r1w00_k30_q1_l75 | 0.744274 |
| r1w00_k40_q1_l75 | 0.742855 |
| r1w00_k10_q1_l50 | 0.740008 |
| r1w00_k20_q1_l65 | 0.739419 |
| r1w00_k20_q6_l25 | 0.736031 |
| r1w00_k30_q1_l65 | 0.731511 |
| r1w00_k30_q6_l25 | 0.731435 |
| r1w00_k10_q1_l40 | 0.731109 |
| r1w00_k40_q1_l65 | 0.728917 |
| r1w00_k40_q6_l25 | 0.724899 |
| r1w00_k20_q1_l50 | 0.718335 |
| r1w00_k10_q1_l25 | 0.716427 |
| r1w00_k40_q1_l50 | 0.704343 |
| r1w00_k20_q1_l40 | 0.697689 |
| r1w00_k30_q1_l50 | 0.696326 |
| r1w00_k40_q1_l40 | 0.684754 |
| r1w00_k30_q1_l40 | 0.670082 |
| r1w00_k20_q1_l25 | 0.653558 |
| r1w00_k40_q1_l25 | 0.638667 |
| r1w00_k30_q1_l25 | 0.617534 |

## Ограничения

Данные уже наблюдались: это development-поиск, не независимая оценка скрытого теста.
Сетка не расширяется после просмотра validation; на validation проверяются лишь baseline и один заранее выбранный вариант.
Результат не переключает активный MVP. Порог и кандидаты должны побайтно совпасть с v24.
Для смешанного ranking в export сохраняются исходные 2048 признаков и точная конфигурация смешивания; NPY replay проверен.
Junk/top-10 и evaluator не меняются. Не используются другие query, камера, время, OCR или правки bbox.
Исходники неизменны: True. Время текущего вызова: 0.9 мин.


Reports and exports: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_25_map_search/runs/map_search_v1
